# Module 4 & 5: Denoising Autoencoder + Baseline Training

Train two models:
1. **Denoising Autoencoder** — reconstructs clean signals from noisy corpus
2. **Letter-Level Baseline** — classifies individual letters (ablation condition ①)

Then compute WER/CER for the baseline condition.

In [ ]:
import os
import sys
import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from models.dataset import get_dataset
from models.dae import DenoisingAutoencoder, DenoisingAutoEncoderTrainer
from models.baseline import LetterClassifier, LetterClassifierTrainer

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✓ Using device: {device}")

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

# Create checkpoints directory
os.makedirs('../checkpoints', exist_ok=True)

## Phase 1: Load Corpus

Load training and validation data from HDF5 corpus.

In [ ]:
# Load corpus
train_loader = []
val_loader = []

with h5py.File('../data/corpus.h5', 'r') as f:
    # Train
    for sample_id in sorted(f['train'].keys()):
        noisy = f['train'][sample_id]['noisy'][:]
        clean = f['train'][sample_id]['clean'][:]
        train_loader.append((noisy, clean))
    
    # Val
    for sample_id in sorted(f['val'].keys()):
        noisy = f['val'][sample_id]['noisy'][:]
        clean = f['val'][sample_id]['clean'][:]
        val_loader.append((noisy, clean))

print(f"✓ Train samples: {len(train_loader)}")
print(f"✓ Val samples: {len(val_loader)}")

## Phase 2: Train Denoising Autoencoder

Train with MSE loss, Adam optimizer, batch size 16, up to 50 epochs with early stopping.

In [ ]:
# Initialize and train
model = DenoisingAutoencoder(use_skip=False)
trainer = DenoisingAutoEncoderTrainer(model, device=device, learning_rate=1e-3)

trainer.fit(
    train_loader,
    val_loader,
    epochs=50,
    batch_size=16,
    patience=7,
    checkpoint_dir='../checkpoints'
)

print("\n✓ DAE training complete")
print(f"  Best checkpoint: ../checkpoints/dae_best.pt")
print(f"  Best val loss: {trainer.best_val_loss:.6f}")

## Phase 3: Train Letter-Level Baseline Classifier

Train on isolated letters (80/10/10 split) for ablation condition ①.

In [ ]:
# Load dataset
dataset = get_dataset("../data/icub_braille_raw")

# Prepare letter data
all_letters = []
for idx, char in enumerate('abcdefghijklmnopqrstuvwxyz '):
    recordings = dataset.get_all_letters(char)
    for recording in recordings:
        all_letters.append((recording, idx))

print(f"Total letter samples: {len(all_letters)}")

# Split 80/10/10
n = len(all_letters)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

indices = np.random.permutation(n)
train_data = [all_letters[i] for i in indices[:n_train]]
val_data = [all_letters[i] for i in indices[n_train:n_train + n_val]]
test_data = [all_letters[i] for i in indices[n_train + n_val:]]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# Train
model = LetterClassifier(num_classes=27, fixed_length=150)
trainer = LetterClassifierTrainer(model, device=device, learning_rate=1e-3)

trainer.fit(
    train_data,
    val_data,
    epochs=50,
    batch_size=32,
    patience=10,
    checkpoint_dir='../checkpoints'
)

# Test
test_acc, per_char_acc = trainer.test(test_data)

print(f"\n✓ Baseline classifier training complete")
print(f"  Overall accuracy: {test_acc * 100:.2f}%")
print(f"  Checkpoint: ../checkpoints/letter_baseline.pt")

## Summary

✓ Module 4 (Denoising Autoencoder) training complete
✓ Module 5 (Letter-Level Baseline) training complete

**Next Step:** Run evaluation to compute WER/CER for condition ①
```bash
python ../evaluate_baseline.py
```